# Retail Sales Data Analysis
This notebook covers data import, cleaning, feature engineering, exploratory analysis, and exporting the cleaned dataset to PostgreSQL.

### Load the Dataset
Load the raw retail sales dataset into a pandas DataFrame.

In [4]:
import pandas as pd

df = pd.read_csv("Retail Sales Dataset.csv")

### Initial Data Inspection
Check the dataset's shape, columns, data types, missing values, and summary statistics.

In [3]:
df.head()

,Order ID,Date,Customer ID,Gender,Age,Product Category,Payment Method,City,Store Type,Quantity,Price per Unit,Total Amount,Total Price,Delivery Status,High Value Order,Sales Rep E-mail
0,559,01/01/2023,CUST559,Female,40,Clothing,Credit Card,Miami,Department Store,4,300,1200,1200,Pending,Yes,miamirep@gmail.com
1,180,01/01/2023,CUST180,Male,41,Clothing,Debit Card,San Francisco,Pharmacy,3,300,900,900,Pending,Yes,sanfranciscorep@gmail.com
2,522,01/01/2023,CUST522,Male,46,Beauty,Debit Card,San Francisco,Specialty Store,3,500,1500,1500,Completed,Yes,sanfranciscorep@gmail.com
3,979,02/01/2023,CUST979,Female,19,Beauty,Mobile Payment,Boston,Convenience Store,1,25,25,25,Completed,No,bostonrep@gmail.com
4,163,02/01/2023,CUST163,Female,64,Clothing,Mobile Payment,Chicago,Pharmacy,3,50,150,150,Pending,No,chicagorep@gmail.com


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Order ID           1000 non-null   int64 
 1   Date               1000 non-null   object
 2   Customer ID        1000 non-null   object
 3   Gender             1000 non-null   object
 4   Age                1000 non-null   int64 
 5   Product Category   1000 non-null   object
 6   Payment Method     1000 non-null   object
 7   City               1000 non-null   object
 8   Store Type         1000 non-null   object
 9   Quantity           1000 non-null   int64 
 10  Price per Unit     1000 non-null   int64 
 11  Total Amount       1000 non-null   int64 
 12  Total Price        1000 non-null   int64 
 13  Delivery Status    1000 non-null   object
 14  High Value Order   1000 non-null   object
 15  Sales Rep E-mail   1000 non-null   object
dtypes: int64(6), object(10)
memory usage: 125.1

In [4]:
df.describe()

,Order ID,Age,Quantity,Price per Unit,Total Amount,Total Price
count,1000.000000,1000.00000,1000.000000,1000.000000,1000.000000,1000.000000
mean,500.500000,41.39200,2.632000,179.890000,476.160000,476.160000
std,288.819436,13.68143,1.465853,189.681356,620.188349,620.188349
min,1.000000,18.00000,1.000000,25.000000,25.000000,25.000000
25%,250.750000,29.00000,1.000000,30.000000,60.000000,60.000000
50%,500.500000,42.00000,3.000000,50.000000,150.000000,150.000000
75%,750.250000,53.00000,4.000000,300.000000,900.000000,900.000000
max,1000.000000,64.00000,10.000000,500.000000,5000.000000,5000.000000


In [5]:
df.isnull().sum()

Order ID             0
Date                 0
Customer ID          0
Gender               0
Age                  0
Product Category     0
Payment Method       0
City                 0
Store Type           0
Quantity             0
Price per Unit       0
Total Amount         0
Total Price          0
Delivery Status      0
High Value Order     0
Sales Rep E-mail     0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

### Data Cleaning
Handle missing values, correct inconsistent text values, and prepare the dataset for analysis.

In [7]:
df = df.drop(['Delivery Status', 'High Value Order', 'Sales Rep E-mail '], axis=1)

In [8]:
# Check if they are identical
(df['Total Amount'] == df['Total Price']).value_counts()

True    1000
Name: count, dtype: int64

In [9]:
df = df.drop(['Total Price'], axis=1)

In [10]:
df.columns = (df.columns.str.strip().str.lower().str.replace(' ', '_'))

In [11]:
invalid_dates = df[pd.to_datetime(df['date'], errors='coerce').isna()]

In [12]:
invalid_dates["date"]

26     13/01/2023
27     13/01/2023
28     13/01/2023
29     13/01/2023
30     14/01/2023
          ...    
993    29/12/2023
994    29/12/2023
995    29/12/2023
996    29/12/2023
997    31/12/2023
Name: date, Length: 586, dtype: object

In [13]:
df["date"].head(10)

0    01/01/2023
1    01/01/2023
2    01/01/2023
3    02/01/2023
4    02/01/2023
5    02/01/2023
6    02/01/2023
7    03/01/2023
8    04/01/2023
9    04/01/2023
Name: date, dtype: object

In [14]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)

In [15]:
df['price_per_unit'] = df['price_per_unit'].astype('float64')
df['total_amount'] = df['total_amount'].astype('float64')

### Feature Engineering
Create new columns or transform existing columns to support better analysis.

In [16]:
df['year']  = df['date'].dt.year
df['month'] = df['date'].dt.month

In [17]:
bins = [0, 18, 25, 35, 45, 55, 100]
labels = ['<18', '18-24', '25-34', '35-44', '45-54', '55+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

In [18]:
df.dtypes

order_id                     int64
date                datetime64[ns]
customer_id                 object
gender                      object
age                          int64
product_category            object
payment_method              object
city                        object
store_type                  object
quantity                     int64
price_per_unit             float64
total_amount               float64
year                         int32
month                        int32
age_group                 category
dtype: object

In [19]:
df.columns

Index(['order_id', 'date', 'customer_id', 'gender', 'age', 'product_category',
       'payment_method', 'city', 'store_type', 'quantity', 'price_per_unit',
       'total_amount', 'year', 'month', 'age_group'],
      dtype='object')

In [20]:
df.isnull().sum()

order_id            0
date                0
customer_id         0
gender              0
age                 0
product_category    0
payment_method      0
city                0
store_type          0
quantity            0
price_per_unit      0
total_amount        0
year                0
month               0
age_group           0
dtype: int64

### Export Cleaned Data to PostgreSQL
Load the cleaned dataset into a PostgreSQL database for further analysis and reporting.

In [22]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

username = "postgres"
password = quote_plus("kajal@202607")
host = "localhost"
port = "5432"
database = "retail_sales_market"

engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)

table_name = "retail_sales"

df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'retail_sales' in database 'retail_sales_market'.


### Conclusion
Summarize the key data preparation steps and confirm that the dataset is ready for EDA and dashboarding.